# Pokémon TCG Battle Agent — Colab Pro Training & Evaluation

Thin Colab notebook wrapping the repository CLI (`python -m src.cli ...`).
- Dataset: `kaggle/the-pokemon-company-ptcg-ai-battle-challenge-playground-episodes-2026-11-13` (Version 1)
- Simulator: CABT Engine (`cg` package)
- Target Runtime: **Colab Pro GPU (NVIDIA T4 / L4 / A100)**
- Click **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

In [ ]:
# 1. Check GPU accelerator and mount Google Drive for persistent checkpoints
!nvidia-smi
import os, torch
print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST_DIR = '/content/drive/MyDrive/ptcg_checkpoints'
except Exception:
    PERSIST_DIR = './checkpoints'
os.makedirs(PERSIST_DIR, exist_ok=True)
print('Persistent checkpoint dir:', PERSIST_DIR)

In [ ]:
# 2. Install dependencies
!pip install -q kagglehub kaggle --no-warn-script-location

In [ ]:
# 3. Download replay dataset (manifest.csv + episodes) and preprocess on CPU
!python -m src.cli download-data --out-dir data/replays --num-top 30
!python -m src.cli preprocess --replay-dir data/replays --out-dir data/processed --seed 42

In [ ]:
# 4. Smoke test simulator on CPU
!python -m src.cli smoke-test --deck deck.csv --seed 42

In [ ]:
# 5. Train baseline imitation learning policy on GPU and save to persistent storage
!python -m src.cli train --data-dir data/processed --checkpoint-dir {PERSIST_DIR} --epochs 20 --batch-size 128 --lr 0.002 --seed 42 --device cuda

# Verify checkpoint resume (+5 epochs) and copy back to local ./checkpoints/
!python -m src.cli train --data-dir data/processed --checkpoint-dir {PERSIST_DIR} --epochs 25 --batch-size 128 --lr 0.002 --seed 42 --device cuda --resume {PERSIST_DIR}/last_model.pt
import shutil
if os.path.abspath(PERSIST_DIR) != os.path.abspath('./checkpoints'):
    os.makedirs('./checkpoints', exist_ok=True)
    for fn in ['best_model.pt', 'last_model.pt', 'train_summary.json']:
        shutil.copy2(os.path.join(PERSIST_DIR, fn), os.path.join('./checkpoints', fn))

In [ ]:
# 6. Evaluate checkpoint on held-out test split and run paired simulator matches
!python -m src.cli evaluate --checkpoint checkpoints/best_model.pt --games 40 --seed 42 --out-file checkpoints/eval_results.json